Welcome! The goal of this notebook is to familiarize economists with the numerical optimization techniques often required for quantitative work, as well as a fairly blatant attempt to sway others towards doing such work in open source software such as Python/Pyomo (or Julia/JuMP, which is faster but I think less flexible in the types of nonlinear problems it can easily adapt to) and combine resources.

In this notebook, I'll introduce some problems you may encounter in classwork and how you would tackle such problems using numerical optimization (even if the problems may have direct solutions), explaining the concepts as they arise.

### Prerequisites 
- Python installation of 2.7 or 3.4+, I recommend Anaconda https://www.anaconda.com/products/individual
- Pyomo, which can be installed from the directions at https://pyomo.readthedocs.io/en/stable/installation.html
- Optimization solvers such as ipopt -- ```conda install -c conda-forge ipopt``` (Note: you may have to debug this, I had to debug a bit on Ubuntu but directions are platform specific). You can install this without Anaconda, but it is extremely complicated. Using conda is **strongly** recommended. 
- Numpy, ```conda install numpy```


### Problem 1. Solving for an optimal consumption bundle / simple nonlinear programming problem

Consider the consumer's maximization problem of 
$$\max_{\{x_i\}_{i=1}^N} U(x_1,\dots,x_n)$$ subject to the standard budget constraint that $$\sum_i p_ix_i \leq w.$$

Suppose the number of goods is N=2.

Let's begin by assuming $U(\cdot)$ is Cobb Douglas with equal shares, i.e. $U(x_1,x_2)=x_1^{1/2}x_2^{1/2}$. Let's also assume that $p_1=2$, $p_2=1$ and $w=10$. Solving for the optimal consumption bundle using standard techniques, $x_1^*=(1/2)\times\frac{w}{p_1}=2.5$ and $x_2^*=5$. But let's pretend for now that this was particularly difficult and we needed to turn to the computer to solve this.


In [1]:
import numpy as np
import pyomo.environ as pyo ### imports pyomo environment
from pyomo.opt import SolverFactory, SolverStatus, TerminationCondition ### imports solvers
opt = SolverFactory('ipopt') ### Free solver good for nonlinear problems

model = pyo.ConcreteModel() ### define an initial model. 
# Note that this model is concrete because we initialize it with real, numeric values for prices
# See: https://pyomo.readthedocs.io/en/stable/pyomo_overview/abstract_concrete.html

N = 2 # number of goods
model.N = N # set this parameter for the model 
model.i = pyo.RangeSet(1, model.N) # create a set for these goods (goods in range 1 to N)

model.alpha = pyo.Param(model.i,within=pyo.NonNegativeReals,default=1.0/N) # set alpha as a vector of parameters 
# alpha is indexed by the set i
# within = fixes the parameter domain to the nonnegative reals
# default: set a default value to 1/N

model.w = 10
model.p = {1:2,2:1} # dictionary of prices
model.x = pyo.Var(model.i,within=pyo.NonNegativeReals,initialize=1.0)
# set variables, indexed by the set i. These are ultimately set by the optimization process
# set domain with within
# give a start value to start the optimization

def CobbDouglasUtility(model):
    # define the utility function, multiplying each x^alpha
    # import the model so we can grab the different objects we defined above from it
    return pyo.prod(model.x[i]**model.alpha[i] for i in model.i)

def BudgetConstraint(model):
    # define the budget constraint, adding up each p*x and setting <= w
    return(sum(model.p[i]*model.x[i] for i in model.i) <= model.w)

# set objective function
model.objective = pyo.Objective(rule=CobbDouglasUtility, sense=-1) ### last argument sets objective to maximize; minimize is the default

# set constraint
model.BC = pyo.Constraint(rule=BudgetConstraint)

# call ipopt to solve
results = opt.solve(model)

In [2]:
if (results.solver.status == SolverStatus.ok) and (results.solver.termination_condition == TerminationCondition.optimal):
     model.display()
elif results.solver.termination_condition == TerminationCondition.infeasible:
     print ("This is an infeasible problem")
else:     # something else is wrong
    print(str(results.solver))

Model unknown

  Variables:
    x : Size=2, Index=i
        Key : Lower : Value              : Upper : Fixed : Stale : Domain
          1 :     0 : 2.5000000232267734 :  None : False : False : NonNegativeReals
          2 :     0 : 5.0000000464532866 :  None : False : False : NonNegativeReals

  Objectives:
    objective : Size=1, Index=None, Active=True
        Key  : Active : Value
        None :   True : 3.535533938780264

  Constraints:
    BC : Size=1
        Key  : Lower : Body               : Upper
        None :  None : 10.000000092906834 :  10.0


Notice here that the optimal values of $x_1$ and $x_2$ are somewhat above the optimal values we calculated. This is because the solver aims to maximize the objective while holding the constraint within some small tolerance of precision. You can adjust this using the parameter ```comparison_tolerance_for_fixed_vars``` (which is set to 1e-05 as a default). In general, adjusting the tolerance would come at the expense of computation time, and I don't foresee the need in most applications to mess around here.

In [3]:
# note that above you can see that the child "x" of "model" has 2 keys,
# 1 and 2. This gives us the hint that x is a dictionary that we can 
# subset just like any other!

x1 = np.round(model.x[1].value,4) ### enough to round off any tolerance errors
x2 = np.round(model.x[2].value, 4)
print("Optimal x1 is",x1,"and optimal x2 is",x2)

Optimal x1 is 2.5 and optimal x2 is 5.0


In [4]:
# getting the max value of the objective function:
model.objective()

3.535533938780264

In [5]:
# getting fulfilled value of the budget constraint:
model.BC()

10.000000092906834

Now, the beauty of Pyomo is that we can easily adjust the parameters of the model and resolve. 

Suppose now that $N=3$, and $U(\cdot)$ is quasilinear, i.e. $U(x_1,x_2,x_3)=x_1+\sqrt{x_2\times x_3}$. And now suppose that $\hat{p} = (2,1,1)$.

In [6]:
model = pyo.ConcreteModel() ### define an initial model
N = 3
model.N = N
model.i = pyo.RangeSet(1, model.N)
model.p = {1:2,2:1,3:1}
model.x = pyo.Var(model.i,within=pyo.PositiveReals,initialize=1.0)
model.w = 10


def QLinearUtility(model):
    return model.x[1]+(model.x[2]*model.x[3])**(1/2)

model.objective = pyo.Objective(rule=QLinearUtility, sense=-1)
model.BC = pyo.Constraint(rule=BudgetConstraint)
results = opt.solve(model)

In [7]:
x1 = np.round(model.x[1].value, 4) ### enough to round off any tolerance errors
x2 = np.round(model.x[2].value, 4)
x3 = np.round(model.x[3].value, 4)

print("Optimal x1 is",x1,"optimal x2 is",x2,"optimal x3 is",x3)

Optimal x1 is 3.1099 optimal x2 is 1.8901 optimal x3 is 1.8901


Notice that in order to change a few parameters and functional forms, we had to create a whole new model. However, Pyomo provides another class which makes changing data quite a bit easier.

In [8]:
N =  3
model = pyo.AbstractModel()

model.N = pyo.Param()
model.i = pyo.RangeSet(1, model.N)
model.p = pyo.Param(model.i,within=pyo.PositiveReals)
model.x = pyo.Var(model.i,within=pyo.PositiveReals,initialize=1.0)
model.w = pyo.Param(default=10)

model.objective = pyo.Objective(rule=QLinearUtility, sense=-1)
model.BC = pyo.Constraint(rule=BudgetConstraint)

Note that the above code is fairly similar to before, except for that we haven't defined any data at the time of creation. We are essentially just naming things that we will later assign to specific values (though we did set a default for w). To do so, we create an instance which will read from a Python dict of the following form:

In [9]:
data = {None: {'N':{None: 3},
               'p':{1:2,2:1,3:1}
              }
       }
instance = model.create_instance(data)
results = opt.solve(instance) ### for more information on solver output, use tee=True

One difference worth noting is that the variable values are now stored in ```instance```. Above, I have overwritten the default value for N to 3, but left alpha and w undefined, as the default value of 10 is fine. By using the initialize and default args above, we can use the AbstractModel() class very similarly to a ConcreteModel(), with the difference being that we need to create an instance first before solving with an AbstractModel(). 

In [10]:
x1 = np.round(instance.x[1].value, 4) ### enough to round off any tolerance errors
x2 = np.round(instance.x[2].value, 4)
x3 = np.round(instance.x[3].value, 4)

print("Optimal x1 is",x1,"optimal x2 is",x2,"optimal x3 is",x3)

Optimal x1 is 3.1099 optimal x2 is 1.8901 optimal x3 is 1.8901


Suppose we again wanted to change the utility function back to Cobb Douglas and change prices to $\hat{p}=(1,1,1)$, then we would simply run:

In [11]:
model.del_component(model.w) ### There's no need to delete w here, I'm just showing how you can
model.objective.deactivate() ### deactivate old objective function which is q-linear

model.w = pyo.Param(default=10)
model.alpha = pyo.Param(model.i,within=pyo.NonNegativeReals,default=1.0/N)
model.new_objective = pyo.Objective(rule=CobbDouglasUtility, sense=-1)

data[None]['p']={1: 1, 2: 1, 3: 1} ### change prices in dict
instance = model.create_instance(data)
results = opt.solve(instance) 

x1 = np.round(instance.x[1].value, 4) ### enough to round off any tolerance errors
x2 = np.round(instance.x[2].value, 4)
x3 = np.round(instance.x[3].value, 4)

print("Optimal x1 is",x1,"optimal x2 is",x2,"optimal x3 is",x3)

Optimal x1 is 3.3333 optimal x2 is 3.3333 optimal x3 is 3.3333


### Problem 2. Solving for parameter values of a workhorse trade model (the Armington model)

Consider the following setup: There are $1,\dots,N$ countries in the world. Consumers in each country consume a varieties of a traded good using CES demand an with elasticity of substitution over goods denoted $\sigma$. Labor is the only factor of production which is inelastically supplied and assumed to be immobile. Trade between countries is costly (iceberg trade costs between i and j are denoted $\tau_{ji} \geq 1$ ) and the no-arbitrade condition applies, meaning that if $p_j$ is the price of a good in country $j$ and $p_{ji}$ is the price of country $j$'s output in country $i$, then we must have that $p_{ji}=\tau_{ji} p_j$. Each country produces a different variety of the good, known as the Armington assumption. Locations are characterized by productivities/marginal products of labor $A_i$. 

The main equation that defines the equilibrium of the Armington model is the goods market clearing condition, which states that each country's revenue from selling to all markets must equal its income:
\begin{equation}
w_jL_j = \sum_{i=1}^N w_iL_i \pi_{ji},
\end{equation}
where $\pi_{ji}$ is the share of expenditure country $i$ spends on goods from country $j$. In short, this says that for each exporter $j$, the sum of what every importer $i$ spends on $j$'s goods (the RHS) must equal $j$'s total income (the LHS).

CES demand tells us that the share of expenditure country $i$ spends on goods from country $j$ is given by
$\pi_{ji} =X_{ji}/Y_i= p_{ji}^{1-\sigma} P_i^{\sigma-1}$, where $P_i^{1-\sigma}=\sum_l p_{li}^{1-\sigma}$. But if production of each good in each country is perfectly competitive, then $p_i=w_i/A_i$. Therefore, the price in country $i$ of consuming one unit from country $j$ is:
$p_{ji}=\tau_{ji}\frac{w_j}{A_j}$. Therefore, $$\pi_{ji}= \frac{\left(\tau_{ji}\frac{w_j}{A_j}\right)^{1-\sigma}}{\sum_l \left(\tau_{li}\frac{w_l}{A_l}\right)^{1-\sigma}}.$$

#### (Jump to here if you only care about the objective statement) This implies that the main equilibrium clearing condition is given by 
\begin{equation}
w_jL_j = \sum_{i=1}^N w_iL_i\frac{\left(\tau_{ji}\frac{w_j}{A_j}\right)^{1-\sigma}}{\sum_l \left(\tau_{li}\frac{w_l}{A_l}\right)^{1-\sigma}}, \forall j=1,\dots,N.
\end{equation}

Note that the price index $P_i^{1-\sigma}=\sum_l \left(\tau_{li}\frac{w_l}{A_l}\right)^{1-\sigma}$ appears inside the sum over importers $i$, since each destination market has its own price index. This means the equation cannot be simplified by multiplying through by a single $P^{1-\sigma}$ term.

Suppose we have information on the wages of workers in each country ($w_i$) and its total population stock ($L_i$), as well as the trade costs between each country pair ($\tau_{ji}$) and the elasticity of substitution ($\sigma=6$), but we don't know the productivities of each country $A_i$. The equilibrium clearing condition above provides us $N$ equations (i.e. one market clearing condition for each country $j$), which should be sufficient to solve for the N unknowns $A_1, \dots, A_N$. But, you'll notice that $A_1, \dots, A_N$ do not admit a closed form solution, and so we need to solve for these values using numerical optimization. These values will $\textbf{not}$ be unique, but they will be unique up to scale, so we can set the sum of the productivities equal to a normalizing constant (the choice of this will be important).

I've skipped alot of details because they're not really important here. But for more information on the setup of this model, see here:
https://scholar.princeton.edu/sites/default/files/zidar/files/connecting_theory_empirics_allen_gravity.pdf

In [12]:
import csv
with open('wages_pop.csv') as f:
    reader = csv.DictReader(f)
    sum_productivities = sum(float(row['wage']) for row in reader)

model = pyo.AbstractModel()

model.i = pyo.Set() # again, name objects
model.w = pyo.Param(model.i) # name wage (function of model.i because there is one per country)
model.L = pyo.Param(model.i) # name labor (function of model.i because there is one per country)
model.tau = pyo.Param(model.i,model.i) # name tau (function of model.i x model.i because there is one per country-pair)

def A_init_rule(model, i):
    """Initialize productivities proportional to wages -- in autarky, A_i ~ w_i."""
    wage_sum = sum(model.w[j] for j in model.i)
    return model.w[i] * sum_productivities / wage_sum

model.A = pyo.Var(model.i,domain=pyo.NonNegativeReals, initialize=A_init_rule) # name A, the VAR we want to solve for, one per country
model.sigma = 6.0

Notice how above we initialize our guess for the $A_i$'s proportional to wages. This is because in autarky (i.e., no trade), $A_i \approx w_i$, so wages provide a reasonable starting neighborhood. The choice of initial values in nonlinear problems will matter a lot. Note that if an initial value (given by initialize or default) is not provided, Pyomo will start by assuming the value is zero, which here will cause there to be zeros in the denominators of our problem.

From Hart et al., <a href="https://link.springer.com/content/pdf/10.1007/978-3-319-58821-6.pdf">Pyomo Optimization Modeling in Python</a>, regarding nonlinear programming problems: """For the general nonconvex case, nonlinear programming problems can, and often do, have multiple local solutions. While academic and commercial solvers do exist that are mathematically guaranteed to find the global solution of a nonlinear programming problem, large problems cannot be solved by state-of-the-art solvers. Consequently, one is often forced to employ a solver that only provides a guarantee of local optimality, for which it is often critical to initialize the problem near the desired local solution."""

I don't have any advice here to give, other than to say that you will have to mess around -- sometimes the model will converge for certain initial values, but not others. Other times, you will simply achieve convergence much faster. For this particular model, the autarky-based initialization above turns out to be important -- it puts us close enough to the solution that Ipopt can take it from there.

In [13]:
def p_ji(model, j, i):
    """Numerator of trade share: (tau_ji * w_j / A_j)^{1-sigma}."""
    return (model.tau[j,i] * model.w[j] / model.A[j]) ** (1 - model.sigma)

# We store the price index P_i as its own variable so Ipopt doesn't have to
# deal with deeply nested sums inside the market clearing constraints.
model.P = pyo.Var(model.i, domain=pyo.PositiveReals, initialize=1.0)

def price_index_def_rule(model, i):
    """Price index definition: P_i = sum_l (tau_li * w_l / A_l)^{1-sigma}."""
    return model.P[i] == sum(p_ji(model, l, i) for l in model.i)

def goods_market_clearing_rule(model, j):
    """Goods market clearing for exporter j: revenue from all markets = income."""
    return model.w[j] * model.L[j] == sum(
        model.w[i] * model.L[i] * p_ji(model, j, i) / model.P[i]
        for i in model.i)

def productivity_sum(model):
    """Normalization: sum of productivities = constant."""
    return sum(model.A[i] for i in model.i) == sum_productivities

# Dummy objective -- the system is square so we're really just solving
# a system of equations; there's nothing to maximize or minimize
model.objective = pyo.Objective(rule=lambda model: 0, sense=1)

# Price index definition (N equations tying P to A)
model.price_index_def = pyo.Constraint(model.i, rule=price_index_def_rule)

# Market clearing (one per exporter)
model.market_clearing = pyo.Constraint(model.i, rule=goods_market_clearing_rule)

# Normalization
model.a_fix = pyo.Constraint(rule=productivity_sum)

What if your data (i.e. parameter values) is stored in a csv file? Read https://pyomo.readthedocs.io/en/stable/working_abstractmodels/data/dataportals.html for more info, but an example is provided below using the two csv files "wages_pop.csv" and "tau.csv"

In [14]:
data = pyo.DataPortal()
data.load(filename='wages_pop.csv', param=(model.w,model.L), index=model.i)
data.load(filename='tau.csv', param=model.tau, format='array')
instance = model.create_instance(data)

# Initialize P[i] from the starting A values so the price index constraints
# are approximately satisfied at start — helps the solver converge
for i in instance.i:
    instance.P[i].set_value(
        sum(pyo.value(instance.tau[l,i] * instance.w[l] / instance.A[l]) ** (1 - 6.0)
            for l in instance.i)
    )

So now we have $2N$ variables ($A_1,\dots,A_N$ and $P_1,\dots,P_N$) and $2N+1$ constraints ($N$ price index definitions, $N$ market clearing conditions, and $1$ normalization). But you'll recall from the theory that one of the market clearing conditions is redundant by Walras' law, so we can drop it -- leaving us with a square system of $2N$ equations in $2N$ unknowns. Since there's nothing to maximize or minimize here (we just want to find values that satisfy the equations), we set the objective to zero and let Ipopt solve the feasibility problem.

In [15]:
# Deactivate one market clearing constraint (Walras' law: N-1 independent conditions)
N = len(list(instance.i))
instance.market_clearing[N].deactivate()

opt.options['max_iter'] = 3000
opt.options['tol']      = 1e-8

results = opt.solve(instance, tee=True)

if (results.solver.status == SolverStatus.ok) and (results.solver.termination_condition == TerminationCondition.optimal):
    print("\nIpopt converged — solution found.")
elif results.solver.termination_condition == TerminationCondition.infeasible:
    print("\nInfeasible — check initialization or model specification.")
else:
    print(f"\n{results.solver.status}, {results.solver.termination_condition}")

Ipopt 3.14.5: max_iter=3000
tol=1e-08


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.5, running with linear solver MUMPS 5.2.1.

Number of nonzeros in equality constraint Jacobian...:    80399
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:    40200

Total number of variables............................:      400
                     variables with only lower bounds:      400
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:

Now, let's look at the solved productivity values and then verify that the solution is reasonable by checking trade balances.

In [16]:
print(f"Sum of productivities: {sum(instance.A[i].value for i in instance.i):.4f}  (target: {sum_productivities:.4f})")

Sum of productivities: 4201.2006  (target: 4201.2006)


In [17]:
for i in range(1,200+1):
    print(np.round(instance.A[i].value,3))

21.883
10.141
25.062
21.68
22.446
10.553
14.627
23.603
12.862
35.016
16.672
15.928
27.087
14.637
38.542
22.363
17.058
15.261
15.18
25.29
18.204
20.281
53.21
20.063
23.596
15.33
15.518
18.318
14.211
19.801
26.799
17.707
45.436
11.028
29.584
27.927
21.754
14.495
22.17
14.733
28.945
14.599
40.748
18.174
20.141
13.596
14.323
12.232
41.502
26.398
34.001
21.712
13.776
13.032
14.99
17.481
16.454
13.12
14.149
17.996
18.59
15.177
15.303
18.747
12.867
19.908
18.712
19.733
21.635
34.899
13.311
42.549
21.084
28.161
17.88
17.357
13.531
10.386
10.589
16.137
11.706
29.797
18.543
11.879
18.981
14.331
14.598
17.564
18.359
27.741
19.883
21.658
41.317
24.111
12.773
14.143
7.747
25.573
8.03
12.046
24.749
8.509
31.583
28.875
42.631
14.033
13.997
12.644
15.462
17.077
10.648
28.758
29.172
24.176
64.076
14.078
9.879
13.36
18.906
23.657
18.709
25.129
26.535
30.221
16.749
15.309
22.415
41.864
28.622
30.865
21.62
30.499
14.826
28.912
12.012
18.565
31.219
19.355
23.401
27.238
23.99
17.416
23.51
17.852
12.425
15.3

Now, let's check what these implied productivities yield for trade balances for each country and verify that the solution is reasonable. If the solver converged, these should all be close to zero.

In [18]:
### Check trade balances using the correct goods market clearing equation
### For each exporter j: revenue from all markets should equal income

def p_ji_check(j, i):
    return (tau[j,i] * w[j] / A[j]) ** (-5.0)

def Pi_check(i):
    return sum(p_ji_check(l, i) for l in iso3)

def trade_balance(j):
    """Revenue of exporter j from all markets, minus income of j."""
    revenue = sum(
        w[i] * L[i] * p_ji_check(j, i) / Pi_check(i)
        for i in iso3)
    return revenue - w[j] * L[j]

### Save stored parameter values to dictionaries
w, L, tau, A = {}, {}, {}, {}
iso3 = list(range(1, 201))
for i in iso3:
    w[i]  = instance.w[i]
    L[i]  = instance.L[i]
    A[i]  = instance.A[i].value
    for j in iso3:
        tau[i,j] = instance.tau[i,j]

Now let's loop over all trade balances for each country.

In [19]:
for i in iso3:
    print(trade_balance(i))

1.3911893859130942e-10
1.614224309776091e-10
6.133049623713305e-11
2.842881485776161e-10
1.0230394309473922e-10
8.911893445429087e-11
2.3509105773200645e-10
1.7779200334189227e-10
2.650946129278964e-10
-1.4975825024521328e-09
2.0960033708661285e-10
2.367279705595138e-10
-1.4636469813922304e-10
2.0616575113763247e-10
-7.97200527813402e-10
2.1823964857503597e-10
3.325828501488104e-10
1.1472289784819623e-10
1.9841106535523068e-10
-8.533618256478803e-12
1.3074874516405544e-10
2.3429258533269604e-10
-1.4367174117069226e-10
2.483000471897867e-10
2.561328926731221e-10
2.3299406848309445e-10
2.441158386545794e-10
1.9669332829153063e-10
2.0537527234409936e-10
3.4719782604497595e-10
-1.9880275203831843e-10
2.994458014882184e-10
-3.674642812256934e-09
2.4283242083811274e-10
-4.425473321134632e-10
-3.752731458916969e-10
1.61778146434699e-10
1.4035572704074184e-10
1.738165167353145e-10
2.362536832833939e-10
-2.2787816078562173e-10
2.4720669955513586e-10
-2.2630217699770583e-09
2.2312285352654726e-1

If the solver converged, the trade balances above should all be very close to zero.

In general, some models will be difficult to solve using the "IPOpt" solver (a free, non-linear solver), and sometimes commercial solvers are much, much better at finding solutions to particular types of large problems. You can use fast commercial solvers instead (such as Cplex, Gurobi, and Knitro), but they are not free. I haven't figured out how to get these locally without paying, but we have an institutional subscription to GAMS that can be run on the ARE server, which includes (demos to some of) these and many more commercial solvers. Pyomo allows you to write your problem as a script readable by GAMS, which allows for use of its more powerful solvers (without having to code in GAMS). Be aware that many solvers require you to start with a small value of your objective function, so you may need to either scale down your variables and/or objective function by a constant, or solve it first locally and then export the problem later. Solvers may also require variable boundaries that others do not. To do so, run:

In [20]:
instance.write("goods_mkt_clearing.gms",io_options={'put_results':'dat'})

('goods_mkt_clearing.gms', 140052106866944)

Another resource that may be helpful is the freely provided <a href="https://neos-server.org/neos/">NEOS server at the University of Wisconsin</a> that allows you to run your model remotely using many commerical solvers. To use the server, you need to write your model to a GAMS file as done above.

Questions? Comments?

Email me: jsayre@ucdavis.edu